# 09 - Model evaluation

Objective: inspect the held-out model comparison, actual-versus-predicted returns, residual behavior, and baseline performance.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics_path = ROOT / "reports" / "tables" / "model_comparison.csv"
validation_path = ROOT / "reports" / "tables" / "model_validation.csv"
predictions_path = ROOT / "data" / "processed" / "combined" / "forecast_predictions.parquet"
metrics = pd.read_csv(metrics_path)
validation = pd.read_csv(validation_path)
predictions = pd.read_parquet(predictions_path)
display(metrics.sort_values("mae"))
model = validation.sort_values("mae").iloc[0]["model"]
selected = predictions.loc[predictions.model.eq(model)].sort_values("date").copy()
selected["residual"] = selected.actual_return - selected.predicted_return
print("Test residual mean/std:", selected.residual.mean(), selected.residual.std())
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(selected.date, selected.actual_return, label="Actual")
axes[0].plot(selected.date, selected.predicted_return, label="Predicted")
axes[0].legend(); axes[0].set_ylabel("Return")
axes[1].scatter(selected.predicted_return, selected.residual, s=12, alpha=0.6)
axes[1].axhline(0, color="black", linewidth=0.8); axes[1].set(xlabel="Predicted return", ylabel="Residual")
fig.tight_layout()
figure_path = ROOT / "reports" / "figures" / "modelling"
figure_path.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path / "reliance_residuals.png", dpi=160)
plt.close(fig)

Interpretation: return MAPE is intentionally omitted because actual returns can be zero or close to zero. A visually similar series is not sufficient evidence that the forecast is accurate.

Conclusion: use quantitative test metrics and inspect residuals; this one chronological split is a limited evaluation.